# Silver Layer: Normalize and Clean Data
## 🧹 Data Normalization Pipeline

**Purpose:** Parse raw JSON from Bronze layer and create clean, normalized tables

**Key Features:**
- Idempotent MERGE operations (can run multiple times safely)
- Long format design (one row per ticker-date combination)
- Data quality validations
- Maintains silver.tickers catalog

**Outputs:**
- `silver.tickers` - Master ticker catalog
- `silver.market_ohlcv` - Normalized OHLCV data

---

### Architecture: Bronze → Silver
```
bronze.api_responses (raw JSON)
          ↓
    [Parse & Validate]
          ↓
silver.market_ohlcv (clean OHLCV)
```

## 2️⃣ Read from Bronze Layer

Read successful API responses from `bronze.api_responses` that haven't been processed yet.

In [ ]:
# Parse JSON - DataBursátil API response format:
# { "data": [ { "date": "2025-12-24", "open": 100.5, "high": 102.0, ... } ] }

# Define schema for parsing (adjust based on actual API response)
ohlcv_schema = ArrayType(StructType([
    StructField("date", StringType()),
    StructField("open", DoubleType()),
    StructField("high", DoubleType()),
    StructField("low", DoubleType()),
    StructField("close", DoubleType()),
    StructField("volume", LongType()),
    StructField("adjusted_close", DoubleType())
]))

# Parse JSON and explode array
parsed_df = bronze_df \
    .withColumn("json_parsed", F.from_json(F.col("response_body"), 
                                          StructType([StructField("data", ohlcv_schema)]))) \
    .withColumn("data_array", F.col("json_parsed.data")) \
    .withColumn("ohlcv_row", F.explode(F.col("data_array"))) \
    .select(
        F.col("ticker"),
        F.to_date(F.col("ohlcv_row.date")).alias("date"),
        F.col("ohlcv_row.open").alias("open"),
        F.col("ohlcv_row.high").alias("high"),
        F.col("ohlcv_row.low").alias("low"),
        F.col("ohlcv_row.close").alias("close"),
        F.col("ohlcv_row.volume").alias("volume"),
        F.coalesce(F.col("ohlcv_row.adjusted_close"), F.col("ohlcv_row.close")).alias("adjusted_close"),
        F.lit("MXN").alias("currency"),
        F.lit("databursatil").alias("source"),
        F.col("response_id").alias("source_record_id")
    )

# Add audit columns
parsed_df = parsed_df \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("last_updated", F.current_timestamp())

print(f"✓ Parsed {parsed_df.count()} OHLCV records")
print("\nSample parsed data:")
parsed_df.orderBy(F.col("date").desc()).show(10)

In [ ]:
# Check if table exists, create if not
try:
    delta_table = DeltaTable.forName(spark, "silver.market_ohlcv")
    print("✓ Table silver.market_ohlcv exists")
except:
    print("⚠️  Table silver.market_ohlcv does not exist, creating...")
    # Run DDL script first or create inline
    validated_df.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("date") \
        .option("overwriteSchema", "true") \
        .saveAsTable("silver.market_ohlcv")
    delta_table = DeltaTable.forName(spark, "silver.market_ohlcv")
    print("✓ Table created")

# Perform MERGE (upsert)
print("\n💾 Merging data into silver.market_ohlcv...")

(delta_table.alias("target")
    .merge(
        validated_df.alias("source"),
        "target.ticker = source.ticker AND target.date = source.date"
    )
    .whenMatchedUpdate(set={
        "open": "source.open",
        "high": "source.high",
        "low": "source.low",
        "close": "source.close",
        "volume": "source.volume",
        "adjusted_close": "source.adjusted_close",
        "last_updated": "source.last_updated"
    })
    .whenNotMatchedInsertAll()
    .execute()
)

print("✓ MERGE complete (idempotent upsert)")

# Show updated statistics
spark.sql("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT ticker) as unique_tickers,
        MIN(date) as earliest_date,
        MAX(date) as latest_date
    FROM silver.market_ohlcv
""").show()

In [ ]:
# Optimize Delta tables for better query performance
print("⚡ Optimizing Delta tables...")

spark.sql("OPTIMIZE silver.market_ohlcv ZORDER BY (ticker)")
print("✓ Optimized silver.market_ohlcv")

# Compute statistics
spark.sql("ANALYZE TABLE silver.market_ohlcv COMPUTE STATISTICS")
print("✓ Updated statistics")

print("\n" + "="*60)
print("✅ SILVER LAYER NORMALIZATION COMPLETE")
print("="*60)

# Final summary
summary = spark.sql("""
    SELECT 
        'market_ohlcv' as table_name,
        COUNT(*) as total_rows,
        COUNT(DISTINCT ticker) as unique_tickers,
        COUNT(DISTINCT date) as unique_dates,
        MIN(date) as earliest_date,
        MAX(date) as latest_date
    FROM silver.market_ohlcv
    
    UNION ALL
    
    SELECT 
        'tickers' as table_name,
        COUNT(*) as total_rows,
        SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as unique_tickers,
        NULL as unique_dates,
        MIN(added_date) as earliest_date,
        MAX(added_date) as latest_date
    FROM silver.tickers
""")

summary.show(truncate=False)

print("\n📌 Next Step: Run 03_feature_engineering.ipynb")

## 7️⃣ Optimization and Summary

In [ ]:
# Get unique tickers from processed data
unique_tickers = validated_df.select("ticker").distinct()

# Prepare ticker catalog records
tickers_catalog = unique_tickers.select(
    F.col("ticker"),
    F.lit(None).cast(StringType()).alias("ticker_name"),
    F.lit("BMV").alias("market"),
    F.lit(None).cast(StringType()).alias("sector"),
    F.lit(True).alias("is_active"),
    F.current_date().alias("added_date"),
    F.lit(None).cast(DateType()).alias("deactivated_date"),
    F.create_map().cast(MapType(StringType(), StringType())).alias("metadata"),
    F.current_timestamp().alias("last_updated")
)

# Create or merge into silver.tickers
try:
    tickers_table = DeltaTable.forName(spark, "silver.tickers")
    
    # Merge: only insert new tickers, don't overwrite existing
    (tickers_table.alias("target")
        .merge(
            tickers_catalog.alias("source"),
            "target.ticker = source.ticker"
        )
        .whenMatchedUpdate(set={"last_updated": "source.last_updated"})
        .whenNotMatchedInsertAll()
        .execute()
    )
    
    print("✓ Updated silver.tickers catalog")
    
except:
    print("⚠️  Creating silver.tickers table...")
    tickers_catalog.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("silver.tickers")
    print("✓ Created silver.tickers")

# Show current tickers
spark.sql("""
    SELECT ticker, market, is_active, added_date
    FROM silver.tickers
    ORDER BY ticker
""").show(20)

## 6️⃣ Update silver.tickers Catalog

Ensure all tickers are registered in the master catalog.

## 5️⃣ Idempotent MERGE to silver.market_ohlcv

Use Delta Lake MERGE for idempotent upsert (can run multiple times safely).

In [ ]:
# Data quality checks
print("🔍 Running data quality validations...\n")

# Remove null values in key columns
clean_df = parsed_df.filter(
    F.col("ticker").isNotNull() &
    F.col("date").isNotNull() &
    F.col("close").isNotNull()
)

initial_count = parsed_df.count()
clean_count = clean_df.count()
removed = initial_count - clean_count

print(f"✓ Null check: Removed {removed} rows with null values")

# Remove duplicates (ticker, date)
dedup_df = clean_df.dropDuplicates(["ticker", "date"])
dup_count = clean_count - dedup_df.count()
print(f"✓ Duplicate check: Removed {dup_count} duplicate rows")

# Validate price ranges (basic sanity check)
validated_df = dedup_df.filter(
    (F.col("open") > 0) &
    (F.col("high") > 0) &
    (F.col("low") > 0) &
    (F.col("close") > 0) &
    (F.col("high") >= F.col("low")) &
    (F.col("high") >= F.col("open")) &
    (F.col("high") >= F.col("close")) &
    (F.col("low") <= F.col("open")) &
    (F.col("low") <= F.col("close"))
)

invalid_prices = dedup_df.count() - validated_df.count()
print(f"✓ Price validation: Removed {invalid_prices} rows with invalid prices")

final_count = validated_df.count()
print(f"\n📊 Final clean records: {final_count} (from {initial_count} initial)")
print(f"   Data quality: {(final_count/initial_count*100):.1f}% pass rate")

## 4️⃣ Data Quality Validation

Apply quality checks before writing to Silver layer.

## 3️⃣ Parse JSON and Extract OHLCV Data

Parse the JSON response body and extract OHLCV fields into structured columns.

In [ ]:
# Read successful responses from bronze layer (last 7 days)
bronze_df = spark.sql("""
    SELECT 
        response_id,
        ticker,
        response_body,
        data_date,
        ingestion_timestamp
    FROM bronze.api_responses
    WHERE is_successful = true
      AND data_date >= current_date() - INTERVAL 7 DAYS
    ORDER BY ingestion_timestamp DESC
""")

print(f"✓ Read {bronze_df.count()} successful API responses from bronze layer")
bronze_df.show(5, truncate=False)

In [ ]:
import json
from datetime import datetime, timezone
from typing import Dict, List, Optional

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

print("✓ Libraries imported")
print(f"✓ Spark version: {spark.version}")
print(f"✓ Notebook started at: {datetime.now(timezone.utc).isoformat()}")

## 1️⃣ Setup and Configuration